<a href="https://colab.research.google.com/github/jiraroj-wir/MUIC-ICCS261-Principles-of-Data-Science/blob/main/EDA_Gravitational-Kernel-Analysis%20-and-Compression_Jiraroj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICCS261 Module 2 · Assignment 2.2: EDA on Your Term-Project Dataset

**Accompanies:** Module 2, Topic 2, *Exploratory Data Analysis (EDA)* slides, and the team lab *EDA Lab: Who Survived the Titanic?*

⏱️ **Time budget.** Each part has a budget in its heading. If a part runs well over, stop, write what you have, and note where you got stuck. A clear "I ran out of time here" beats an unfinished notebook.

> "If machine learning and hypothesis testing are where you answer your questions, EDA is where you figure out what questions to ask in the first place." (EDA slides, 01 Foundations of EDA)

In the team lab, the question, the dataset and the checkpoints were given to you. Here you bring your own: the research question you finalised in the peer-feedback activity, and the dataset you chose for your term project. By the end you will know what your data can answer, what it cannot, and which hypothesis you will test in Topics 3 and 4.

**Completed by:** ______________________ &nbsp;&nbsp; **Date:** ____________

## 🗺️ Assignment map

| Part | What you do | Team lab exercise | Slide section | Budget |
| --- | --- | --- | --- | --- |
| 0 | Load, tidy, configure, data dictionary | Ex. 0, Ex. 1 | 02 Data types and structure | 50 min |
| A | Domain and provenance | 08_EDA Task A | 01 Foundations of EDA | 25 min |
| B | Missing values and data quality | Ex. 1 (task 3), Ex. 7 (`deck`) | Lecture 7: Missing data | 30 min |
| C | Numeric variables: location, variability, shape | Ex. 2, 3, 4 | 03 Estimates of location · 04 Exploring data distribution | 45 min |
| D | Categorical variables | Ex. 5 | 04 Visualizing the categorical data distribution | 15 min |
| E | Outliers | Ex. 4 · 08_EDA Task D | 03 Types of outliers · Reading a boxplot | 20 min |
| F | Relationships and a confounder check | Ex. 6, 7 | 05 Correlation · 06 Exploring two or more variables | 60 min |
| G | Data story and verdict on your question | Ex. 8 · 08_EDA Task E | 07 Wrapping up · Data visualization in Python | 40 min |
| H | AI log and reflection | 08_EDA Task F | (this notebook) | 15 min, log kept as you go |
| | | | **Total** | **5 hours** |

### Symbols (same as the team lab, plus two new ones)

| Symbol | Meaning |
| --- | --- |
| 📑 | Slides to review before the part |
| 🎯 | Tasks: code you must write or complete. Replace every `___` |
| ✍️ | **Interpret** box. 🚫 **No AI.** Your own words, with at least one number from your output |
| 🔎 | **Your sanity check.** In the lab you had ✅ checkpoints. Nobody knows the right numbers for your data, so you write a check that shows a key number is plausible |
| 🤖 | A good moment to use AI, and to add a row to your AI log (Part H) |

### Colab tips

- *File → Save a copy in Drive* first. Name it `EDA_[ProjectShortName]_[FirstName].ipynb`.
- Run cells top to bottom. Before you submit: *Runtime → Restart session and run all*. It must finish with no errors.
- Reusing code from the team lab is encouraged. Add a comment saying where it came from.

## 🤖 AI in this assignment: the short version

The full rules and the reasons for them are in the assignment brief. The one rule behind them: **you must be able to explain every line of code and every claim in this notebook without the AI open.**

| Zone | Where | What you record |
| --- | --- | --- |
| 🚫 No AI | Part A, every ✍️ box, Part G, Part H reflection | Nothing: write it yourself |
| 🤖 AI allowed, logged | Code for loading, cleaning and plotting; debugging; explaining a concept or function | A row in the AI log (Part H) |
| 🔍 AI as Skeptic | After you write your Part F interpretation | Exact prompt, the AI's objections, your response |

**Five habits that make AI help you learn** (the brief has example prompts):

1. **Ask for an explanation before code.** "Explain MAD vs standard deviation with one extreme value. No code."
2. **Predict, then run.** Write `# expect: ...` above AI-written code before running it.
3. **Share the schema, not the data.** Paste `df.info()` and `df.head(3)`, never the raw file. Any number the AI states about *your* data is invented; only your outputs count.
4. **Ask for the cause before the fix.** Paste the full traceback, ask "What is the cause?", and write the cause in your own words.
5. **Check the docs.** AI often suggests deprecated calls (e.g., `sns.distplot`). Open the [pandas](https://pandas.pydata.org/docs/) or [seaborn](https://seaborn.pydata.org/api.html) page for any function you have not used in class.

---
# Part 0 · Get your data into one tidy table (50 min)

📑 **Slides:** "Rectangular data structure", "Non-rectangular data structures", "Key data types", "Importance of data typing" · Notebook 06 (tidy data)

✅ **Part 0 checkpoint (due [date]):** your data loads, the configuration check cell prints "Configuration OK", and Sanity check 0 passes. Submit the Colab link on Moodle. If you cannot get here, come to office hours before the checkpoint date, not after.

Everything after this part needs one pandas DataFrame where **one row is one unit of analysis** and **one column is one variable**. The Titanic arrived like that. Your data may not. You do **not** need a data structures or databases course for this part: the patterns below cover almost every project in this class.

In [ ]:
# Setup: run me first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from math import ceil

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)

### 0.1 Load your file

Put your data file in a Drive folder, then mount Drive so the notebook still runs next week.

| Your data looks like | Load it with | Watch out for |
| --- | --- | --- |
| One `.csv` file | `pd.read_csv(PATH + FILE)` | Thai or other non-English text: add `encoding="utf-8-sig"`. Numbers with commas: add `thousands=","` |
| An Excel file | `pd.read_excel(PATH + FILE, sheet_name="Sheet1")` | Title rows above the header: add `skiprows=n` |
| Several files (one per year, region…) | Read each, add a column saying where it came from, then `pd.concat([...], ignore_index=True)` (example below) | Column names must match across files |
| A `.json` file or API response | `pd.json_normalize(data)` | A list inside each record (e.g., reviews inside a product): ask for help |
| A Google Sheet | *File → Download → CSV*, then load as CSV | Record the download date; the sheet may change |
| A database file (`.db`, `.sqlite`) | `pd.read_sql("SELECT * FROM table_name", sqlite3.connect(PATH + FILE))` | You need only this one line. Several tables to join? Ask first |

**Very large file** (over ~500,000 rows or ~200 MB)? After loading, work on a random sample: `df_raw = df_raw.sample(50_000, random_state=42)`, and say so in 0.2.

**Stuck for more than 30 minutes?** Post a specific ask on the *Peer notes* tab or come to office hours. Getting the table right is the hardest part for many projects.

In [ ]:
# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 🎯 Load your data
PATH = "/content/drive/MyDrive/___/"     # the folder that holds your file
FILE = "___"                              # e.g. "survey_2025.csv"

df_raw = pd.read_csv(PATH + FILE)         # change to read_excel / read_json / read_sql if needed (table above)

In [ ]:
# (Only if you have several files) Example: one CSV per year, stacked into one table
# import glob
# parts = []
# for f in sorted(glob.glob(PATH + "*.csv")):
#     part = pd.read_csv(f)
#     part["source_file"] = f.split("/")[-1]   # remember where each row came from
#     parts.append(part)
# df_raw = pd.concat(parts, ignore_index=True)

🎯 **First look** (Lab Ex. 0): show the first rows, a random sample, the shape, and `info()`.

In [ ]:
display(df_raw.head())
display(df_raw.sample(5, random_state=42))
print(df_raw.shape)
df_raw.info()

### 0.2 Configure the notebook for your project

The rest of the notebook reads these settings, so most later cells run without changes. Choose columns that relate to your research question. With many columns, pick **8 to 12** and justify the choice in 0.3.

- `OUTCOME`: the variable your research question is about.
- `GROUP_COL`: your main explanatory or comparison variable.
- `CONFOUNDER_COL`: a third variable that could explain a link between the two (you will check it in Part F). Make your best guess now; you can change it later.

In [ ]:
# 🎯 Configure (replace every ___ ; use [] for an empty list, None for "not applicable")
ID_COL           = ___        # column that identifies one row, e.g. "student_id"; None if there is none
OUTCOME          = "___"      # the variable your research question is about
OUTCOME_TYPE     = "___"      # "numeric" or "categorical"
OUTCOME_POSITIVE = ___        # categorical outcome only: the value you count as a "yes", e.g. 1 or "Pass"; None if numeric
GROUP_COL        = "___"      # main explanatory / comparison variable
CONFOUNDER_COL   = "___"      # a possible confounder for Part F

NUMERIC_COLS     = [___]      # numeric variables to explore (include OUTCOME if it is numeric)
CATEGORICAL_COLS = [___]      # categorical variables to explore (include OUTCOME if it is categorical)

MIN_N = 30                    # groups smaller than this are flagged as "too small to trust"

In [ ]:
# Checks your configuration. If this cell raises an error, fix the cell above.
_all = NUMERIC_COLS + CATEGORICAL_COLS + [OUTCOME, GROUP_COL, CONFOUNDER_COL] + ([ID_COL] if ID_COL else [])
missing_cols = sorted({c for c in _all if c not in df_raw.columns})
assert not missing_cols, f"These names are not columns in df_raw: {missing_cols}"
assert OUTCOME_TYPE in ("numeric", "categorical"), 'OUTCOME_TYPE must be "numeric" or "categorical"'
if OUTCOME_TYPE == "categorical":
    assert OUTCOME_POSITIVE in set(df_raw[OUTCOME].dropna().unique()), "OUTCOME_POSITIVE is not a value of OUTCOME"
non_numeric = [c for c in NUMERIC_COLS if not pd.api.types.is_numeric_dtype(df_raw[c])]
assert not non_numeric, f"Stored as text, not numbers: {non_numeric}. Convert them in 0.4 first, or move them to CATEGORICAL_COLS."

keep = list(dict.fromkeys(([ID_COL] if ID_COL else []) + NUMERIC_COLS + CATEGORICAL_COLS + [OUTCOME, GROUP_COL, CONFOUNDER_COL]))
df = df_raw[keep].copy()
print(f"Configuration OK. Working table: {df.shape[0]:,} rows × {df.shape[1]} columns")

### 0.3 Your unit of analysis ✍️ 🚫 No AI

| Item | Your answer |
| --- | --- |
| My research question (final version, copied exactly) | |
| Data source: name, URL, date downloaded | |
| One row of my DataFrame represents… | |
| Rows × columns (raw → working table) | |
| Sample? (if you used `.sample()`, the size and why) | |
| Columns I kept, and why these | |
| Columns I dropped, and why (IDs, free text, duplicates, unrelated) | |

🔎 **Sanity check 0: prove your unit-of-analysis claim.** If one row is one student, `df[ID_COL].is_unique` must be `True`. If there is no ID column, check for duplicate rows on the columns that should identify a row. If the check fails, your table is not tidy yet: fix it before going on.

In [ ]:
# 🔎 Sanity check 0
if ID_COL:
    print("ID unique:", df[ID_COL].is_unique)
else:
    print("Fully duplicated rows:", df.duplicated().sum())

### 0.4 Data types and the data dictionary

📑 **Slides:** "Key data types", "Importance of data typing" · 08_EDA, *Note on pandas data types*

The cell below drafts your dictionary from the data. Copy it into the markdown table that follows and complete the **Data type (slides)**, **Unit** and **Meaning** columns. If your source publishes a codebook, link it.

In [ ]:
dictionary = pd.DataFrame({
    "pandas dtype": df.dtypes.astype(str),
    "non-null": df.notna().sum(),
    "% missing": df.isna().mean().mul(100).round(1),
    "unique values": df.nunique(),
    "example": df.iloc[0],
})
dictionary

In [ ]:
# 🎯 Fix data types (Lab Ex. 1: "pclass is stored as an integer. Does that make it numeric?")
# Uncomment and adapt the lines you need.

# Categorical stored as numbers or text -> category
# for col in CATEGORICAL_COLS:
#     df[col] = df[col].astype("category")

# Ordinal scale with a meaningful order (e.g. Likert, education level, shirt size)
# df["___"] = pd.Categorical(df["___"], categories=["Low", "Medium", "High"], ordered=True)

# Numbers stored as text (e.g. "1,234" or "45%")
# df["___"] = pd.to_numeric(df["___"].astype(str).str.replace(",", "").str.rstrip("%"), errors="coerce")

# Dates
# df["___"] = pd.to_datetime(df["___"], errors="coerce")

df.dtypes

✍️ **Data dictionary** (🚫 No AI)

| Column | pandas dtype | Data type (slides): continuous / discrete / binary / nominal / ordinal | Unit | Meaning (one line) |
| --- | --- | --- | --- | --- |
| | | | | |
| | | | | |

---
# Part A · Domain and provenance (25 min) 🚫 No AI

📑 **Slides:** 01 Foundations of EDA · 08_EDA Task A

As Task A did for 1912, explain the world your data comes from. Answer in **120 to 180 words**, using these four headings.

1. **Context.** What process produced these rows (survey, sensor, transaction log, census, game server…)? When and where?
2. **Who recorded it, and why.** What was the recorder's own purpose, and how might it shape what was measured? (Update your Assignment 1.5 answer, Part C2, now that you have the real data.)
3. **Who is missing.** Which people, places or events are absent from the record, and how could that bend your answer? (The Titanic data has no crew.)
4. **Proxies.** Which columns stand in for something you cannot measure directly, as `pclass` and `fare` stood in for social status? What does each proxy miss?

✍️ **Your answer:**

(write here)

---
# Part B · Missing values and data quality (30 min)

📑 **Slides / notes:** Lecture 7 *Missing Data* (MCAR, MAR, MNAR; single imputation) · Lab Ex. 1 task 3 and Ex. 7 (is `deck` missing at random?)

🎯 **Tasks**

1. Percentage of missing values per column, sorted from most to least.
2. Number of fully duplicated rows.
3. Check at least one column for impossible values (negative ages, percentages over 100, future dates, scores above the maximum).
4. **Missingness test:** for the column with the most missing values that you plan to keep, compare your outcome between rows where that column is missing and rows where it is not.
5. Record every cleaning decision in the cleaning log, with row counts before and after.

In [ ]:
# Tasks 1-2
missing_pct = df.isna().mean().mul(100).sort_values(ascending=___)
display(missing_pct.round(1).to_frame("% missing"))
print("Fully duplicated rows:", df.duplicated().sum())

In [ ]:
# Task 3: impossible values. Start from the min and max of each numeric column...
display(df[NUMERIC_COLS].agg(["min", "max"]).T)

# ...then write at least one explicit check for your domain, e.g. df[df["age"] < 0]
check_1 = df[___]
print("Rows failing the check:", len(check_1))

In [ ]:
# Task 4: is the missingness related to the outcome? (Lab Ex. 7: survival was 67% with deck known vs 30% missing)
MISSING_COL = "___"   # the column with the most missing values that you plan to keep

is_missing = df[MISSING_COL].isna().rename(f"{MISSING_COL} missing")
if OUTCOME_TYPE == "numeric":
    display(df.groupby(is_missing)[OUTCOME].agg(["count", "mean", "median"]))
else:
    display(pd.crosstab(is_missing, df[OUTCOME], normalize="index").round(3))
    print(pd.crosstab(is_missing, df[OUTCOME], margins=True))

In [ ]:
# Task 5: cleaning log. Wrap every change that drops or alters rows in clean_step().
cleaning_log = []

def clean_step(before, after, what, why):
    cleaning_log.append({"step": what, "why": why, "rows before": len(before), "rows after": len(after)})
    return after

# Examples: adapt or delete
# df = clean_step(df, df.drop_duplicates(), "drop exact duplicates", "same record entered twice")
# df = clean_step(df, df[df["age"].between(0, 110) | df["age"].isna()], "remove impossible ages", "data-entry errors")
# df = clean_step(df, df.assign(region=df["region"].fillna("Unknown")), "region NaN -> 'Unknown'", "missing is informative (MNAR?)")

pd.DataFrame(cleaning_log)

✍️ **Interpret** (🚫 No AI)

- Is the missingness in your most-missing column likely **MCAR, MAR or MNAR**? Use your Task 4 output as evidence.
- For each column with missing values: drop the column, drop the rows, impute (which method?), or keep "Unknown"? Give one reason per decision.
- What did the impossible-value checks find, and what did you do?

(write here)

🔎 **Your sanity check:** e.g., after cleaning, show that `rows after` in the last log entry equals `len(df)`, or that no impossible values remain.

In [ ]:
# 🔎 Sanity check B

---
# Part C · Numeric variables: location, variability, shape (45 min)

📑 **Slides:** 03 "Estimates of location", "Comparing mean, median, mode", "The Bill Gates effect", "Variability (dispersion)", "Standard deviation and variance", "Robust estimates of variability", "Estimates based on percentiles", "Summary statistics in pandas.DataFrame (1/4–4/4)" · 04 "Frequency tables", "Histograms & density plots", "Reading a boxplot", "Creating multiple subplots in one figure"

🎯 **Tasks**

1. One summary table for all of `NUMERIC_COLS`: count, mean, median, 10% trimmed mean, std, IQR, MAD, skewness.
2. **Your own Bill Gates effect:** for the variable where mean and median disagree most, find the rows that pull the mean, and recompute both without them.
3. A histogram with a density curve and a boxplot for every numeric variable.
4. **Bin size matters:** for one variable, three histograms with different numbers of bins, side by side.

In [ ]:
# Task 1: classical and robust estimates side by side (Lab Ex. 2-3, using agg() as on the slides)
def iqr(s):  return s.quantile(.75) - s.quantile(.25)
def mad(s):  return stats.median_abs_deviation(s.dropna())
def trim10(s): return stats.trim_mean(s.dropna(), 0.10)
def skew(s): return s.skew()
for f, name in [(iqr, "IQR"), (mad, "MAD"), (trim10, "trimmed mean 10%"), (skew, "skewness")]:
    f.__name__ = name

summary = df[NUMERIC_COLS].agg(["count", "mean", trim10, "median", "std", iqr, mad, skew]).T
summary["mean ÷ median"] = summary["mean"] / summary["median"].replace(0, np.nan)   # blank when the median is 0
summary.round(2)

In [ ]:
# Task 2: your own Bill Gates effect
VAR = "___"                      # the variable whose mean and median disagree most
cut = df[VAR].quantile(___)      # e.g. 0.99 drops the top 1%; choose and justify
display(df.nlargest(5, VAR))     # who are the extreme rows?

rest = df.loc[df[VAR] <= cut, VAR]
print(f"mean   with / without: {df[VAR].mean():.2f} / {rest.mean():.2f}")
print(f"median with / without: {df[VAR].median():.2f} / {rest.median():.2f}")
print(f"rows removed: {len(df[VAR].dropna()) - len(rest)}")

In [ ]:
# Task 3: distribution of every numeric variable (adapted from 08_EDA, Step 2.2.1)
fig, axes = plt.subplots(len(NUMERIC_COLS), 2, figsize=(11, 3 * len(NUMERIC_COLS)), squeeze=False)
for i, col in enumerate(NUMERIC_COLS):
    sns.histplot(data=df, x=col, kde=True, ax=axes[i, 0])
    axes[i, 0].axvline(df[col].mean(), color="red", ls="--", label="mean")
    axes[i, 0].axvline(df[col].median(), color="green", ls="--", label="median")
    axes[i, 0].legend()
    sns.boxplot(data=df, x=col, ax=axes[i, 1], color="lightcoral")
plt.tight_layout()

In [ ]:
# Task 4: bin size matters (Lab Ex. 4)
BIN_VAR = "___"
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, b in zip(axes, [___, ___, ___]):          # e.g. too few, about right, too many
    ax.hist(df[BIN_VAR].dropna(), bins=b, edgecolor="white")
    ax.set_title(f"{BIN_VAR}, {b} bins")
    ax.set_xlabel(BIN_VAR)
axes[0].set_ylabel("Count")
plt.tight_layout()

✍️ **Interpret** (🚫 No AI). For each numeric variable (a short paragraph or a table):

- What is a **typical value**, and which estimate do you report for it? Use the slides' "use when…" guidance.
- How **spread out** is it? Which measure of spread do you trust, and why?
- Is it **symmetric or skewed**? What does that shape mean in your domain?
- Your **Bill Gates effect:** which rows pulled the mean, and are they a problem or the most interesting part of the data?
- Which bin count did you keep, and what did the others hide or exaggerate?

(write here)

🔎 **Your sanity check:** e.g., the median lies between the min and max for every variable, or a published total or average from your source matches your number.

In [ ]:
# 🔎 Sanity check C

---
# Part D · Categorical variables (15 min)

📑 **Slides:** 04 "Visualizing the categorical data distribution", "Expected value", "Examples of EV calculation", "Be aware of pie charts"

🎯 **Tasks**

1. Counts, proportions and mode of every variable in `CATEGORICAL_COLS`.
2. A **sorted** bar chart for each, with counts on the bars. No pie charts with more than 3 slices.
3. Flag categories with fewer than `MIN_N` rows. Decide whether to merge them, and into what.
4. *(Optional)* If you have a discrete numeric variable (counts such as `sibsp`), compute its expected value Σ x·P(x) and compare it with the mean (Lab Ex. 5).

In [ ]:
# Task 1
for col in CATEGORICAL_COLS:
    table = pd.DataFrame({"count": df[col].value_counts(dropna=False),
                          "proportion": df[col].value_counts(normalize=___, dropna=False).round(3)})
    print(f"\n{col.upper()}   mode = {df[col].mode().iloc[0]}   unique = {df[col].nunique()}")
    display(table)

In [ ]:
# Task 2: sorted bar charts (adapted from 08_EDA, Step 2.2.2)
n_cols = 3
n_rows = ceil(len(CATEGORICAL_COLS) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.8 * n_rows), squeeze=False)
axes_flat = axes.flatten()
for ax, col in zip(axes_flat, CATEGORICAL_COLS):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, color="steelblue")
    ax.bar_label(ax.containers[0])
    ax.set_title(col, fontweight="bold")
    ax.set_ylabel("")
for ax in axes_flat[len(CATEGORICAL_COLS):]:
    fig.delaxes(ax)
plt.tight_layout()

In [ ]:
# Task 3: small categories
for col in CATEGORICAL_COLS:
    counts = df[col].value_counts()
    small = counts[counts < MIN_N]
    if len(small):
        print(f"{col}: {len(small)} categories with n < {MIN_N} -> {small.to_dict()}")

# To merge small categories into "Other" (adapt, and log it with clean_step):
# small_cats = df["___"].value_counts().loc[lambda s: s < MIN_N].index
# df = clean_step(df, df.assign(**{"___": df["___"].astype("object").where(~df["___"].isin(small_cats), "Other")}),
#                 "merge rare categories of ___ into Other", f"n < {MIN_N}")

In [ ]:
# Task 4 (optional): expected value of a discrete numeric variable
# EV_VAR = "___"
# p = df[EV_VAR].value_counts(normalize=True).sort_index()
# print("EV:", (p.index * p).sum(), "| mean:", df[EV_VAR].mean())

✍️ **Interpret** (🚫 No AI)

- Is there a **dominant category** in any variable? What does that imply for comparisons between groups?
- Which categories are **too small** to trust, and what did you do about them?
- Did any variable turn out to be stored as a number but really categorical (like `pclass`)?

(write here)

🔎 **Your sanity check:** e.g., proportions of each variable sum to 1, or category counts sum to `len(df)`.

In [ ]:
# 🔎 Sanity check D

---
# Part E · Outliers (20 min)

📑 **Slides:** 03 "Types of outliers", 04 "Reading a boxplot" · 08_EDA Step 3 and Task D · Lab Ex. 4 ("116 outliers is 13% of passengers")

🎯 **Tasks**

1. For each numeric variable, flag outliers with **both** the 1.5 × IQR rule and |z| > 3. One table: count and % per method.
2. Show the most extreme rows of the variable with the most flagged values.
3. For each flagged group decide: **error** (fix/remove), **legitimate extreme** (keep), or **unsure** (keep, and report key statistics with and without).

In [ ]:
# Task 1
rows = []
for col in NUMERIC_COLS:
    s = df[col].dropna()
    q1, q3 = s.quantile([.25, .75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    n_iqr = ((s < lo) | (s > hi)).sum()
    n_z = (np.abs(stats.zscore(s)) > ___).sum()
    rows.append({"variable": col, "IQR fences": f"[{lo:.2f}, {hi:.2f}]",
                 "IQR outliers": n_iqr, "IQR %": 100 * n_iqr / len(s),
                 "|z|>3 outliers": n_z, "z %": 100 * n_z / len(s)})
outliers = pd.DataFrame(rows).set_index("variable").round(1)
outliers

In [ ]:
# Task 2: look at the actual rows before you decide anything
OUT_VAR = "___"
df.nlargest(8, OUT_VAR)        # use df.nsmallest(...) for low-end outliers

✍️ **Interpret** (🚫 No AI)

- Where do the IQR and z-score methods **disagree**, and why? (Hint: think about skew.)
- Complete the decision table:

| Variable | Flagged rows (n, %) | Error / Legitimate / Unsure | Evidence for your decision | Action |
| --- | --- | --- | --- | --- |
| | | | | |

- If you removed or capped anything, log it with `clean_step()` and show its effect on the mean and median.

(write here)

In [ ]:
# 🔎 Sanity check E (e.g., mean and median of OUT_VAR before vs after your decision)

---
# Part F · Relationships and a confounder check (60 min)

📑 **Slides:** 05 "Understanding correlation", "The correlation coefficient (Pearson's)", "Visualizing correlation", "Limitations and alternatives", "The overplotting problem" · 06 "Overview of multivariate analysis", "Analyzing two categorical variables", "Categorical vs. numeric data", "Visualizing multiple variables" · Data visualization in Python: `heatmap()`, `pairplot()`

This is the first look at the relationship your research question asks about. Choose the tool by the **types** of the two variables (slides: "the appropriate analysis method depends on the nature of the data"):

| Variable 1 × Variable 2 | Statistic | Chart |
| --- | --- | --- |
| numeric × numeric | Pearson **and** Spearman | Scatterplot; hexbin above ~5,000 rows |
| numeric × categorical | Median, IQR and **n** per group | Boxplot + violin plot |
| categorical × categorical | Crosstab: counts and row % | Bar chart of rates by group |

🎯 **Tasks**

1. Pearson and Spearman correlation heatmaps of your numeric variables.
2. Your outcome against your main explanatory variable, and against **one** other variable, using `explore_pair()`.
3. **Confounder check** (Lab Ex. 7, the Cherbourg trap): stratify the outcome-vs-`GROUP_COL` comparison by `CONFOUNDER_COL`, with group sizes.
4. *(Optional)* A `pairplot` of up to five numeric variables.

In [ ]:
# Task 1: correlation heatmaps (Lab Ex. 6)
if len(NUMERIC_COLS) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, method in zip(axes, ["pearson", "spearman"]):
        corr = df[NUMERIC_COLS].corr(method=method)
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
        ax.set_title(f"{method.title()} correlation")
    plt.tight_layout()
else:
    print("Fewer than two numeric variables: skip the heatmap and explain why in your interpretation.")

In [ ]:
# A helper that picks the right statistic and chart for two variables (you may read and modify it)
def _is_num(col):
    return col in NUMERIC_COLS and pd.api.types.is_numeric_dtype(df[col])

def explore_pair(data, x, y):
    d = data[[x, y]].dropna()
    print(f"{x} × {y}   (n = {len(d):,} rows with both values)")
    if _is_num(x) and _is_num(y):
        print(f"Pearson r = {d[x].corr(d[y]):.3f} | Spearman rho = {d[x].corr(d[y], method='spearman'):.3f}")
        if len(d) > 5000:
            sns.jointplot(data=d, x=x, y=y, kind="hex", gridsize=30)
        else:
            sns.scatterplot(data=d, x=x, y=y, alpha=0.5)
    elif _is_num(x) or _is_num(y):
        num, cat = (x, y) if _is_num(x) else (y, x)
        table = d.groupby(cat, observed=True)[num].agg(n="count", median="median", IQR=iqr, mean="mean")
        table["n < MIN_N"] = table["n"] < MIN_N
        display(table.round(2))
        order = table.sort_values("median").index
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        sns.boxplot(data=d, x=cat, y=num, order=order, ax=axes[0])
        sns.violinplot(data=d, x=cat, y=num, order=order, ax=axes[1], cut=0)
        for ax in axes:
            ax.tick_params(axis="x", rotation=30)
        plt.tight_layout()
    else:
        display(pd.crosstab(d[x], d[y], margins=True))
        rates = pd.crosstab(d[x], d[y], normalize="index").round(3)
        display(rates)
        if y == OUTCOME and OUTCOME_TYPE == "categorical":
            rate = rates[OUTCOME_POSITIVE].sort_values()
            ax = rate.plot.barh(color="steelblue")
            ax.set_xlabel(f"Share with {OUTCOME} = {OUTCOME_POSITIVE}")
    plt.show()

In [ ]:
# Task 2: outcome vs your main explanatory variable...
explore_pair(df, GROUP_COL, OUTCOME)

In [ ]:
# ...and vs one other variable (copy this cell if you want to explore more)
explore_pair(df, "___", OUTCOME)

In [ ]:
# Task 3: confounder check (Lab Ex. 7: "Does the Cherbourg advantage survive?")
strat = df[[GROUP_COL, CONFOUNDER_COL, OUTCOME]].dropna().copy()

# A numeric confounder is cut into 3 equal-size groups so it can be used for stratifying
if _is_num(CONFOUNDER_COL):
    strat[CONFOUNDER_COL] = pd.qcut(strat[CONFOUNDER_COL], q=3, duplicates="drop")
# A numeric GROUP_COL is treated the same way
if _is_num(GROUP_COL):
    strat[GROUP_COL] = pd.qcut(strat[GROUP_COL], q=3, duplicates="drop")

if OUTCOME_TYPE == "categorical":
    strat["y"] = (strat[OUTCOME] == OUTCOME_POSITIVE).astype(int)
    label = f"share {OUTCOME} = {OUTCOME_POSITIVE}"
else:
    strat["y"] = strat[OUTCOME]
    label = f"mean {OUTCOME}"

print(f"(a) Overall, by {GROUP_COL}:")
display(strat.groupby(GROUP_COL, observed=True)["y"].agg(["count", "mean"]).round(3))

print(f"(b) Who is in each {GROUP_COL} group? Row % of {CONFOUNDER_COL}:")
display(pd.crosstab(strat[GROUP_COL], strat[CONFOUNDER_COL], normalize="index").round(2))

print(f"(c) {label}, within each level of {CONFOUNDER_COL}:")
display(strat.pivot_table(index=GROUP_COL, columns=CONFOUNDER_COL, values="y", aggfunc="mean", observed=True).round(3))
print("... and the n behind each cell:")
display(strat.pivot_table(index=GROUP_COL, columns=CONFOUNDER_COL, values="y", aggfunc="count", observed=True))

sns.catplot(data=strat, x=GROUP_COL, y="y", col=CONFOUNDER_COL, kind="bar", height=3.5, errorbar=None)
plt.show()

In [ ]:
# Task 4 (optional)
# sns.pairplot(df[NUMERIC_COLS[:5] + ([OUTCOME] if OUTCOME_TYPE == "categorical" else [])],
#              hue=OUTCOME if OUTCOME_TYPE == "categorical" else None, corner=True)

✍️ **Interpret** (🚫 No AI). Write this **before** the Skeptic step below.

- Describe the **direction and strength** of each relationship with numbers. Where Pearson and Spearman differ, why?
- After stratifying by `CONFOUNDER_COL`, **how much of the relationship remains**? Which cells have too few rows to trust?
- What other variable might still explain the pattern, and could you check it with this data?
- Use "**associated with**" or "**differ**", not "causes" or "affects", unless your data comes from an experiment.

(write here)

### 🔍 AI as Skeptic

In the team lab, the Skeptic asked "How do we know? What's n? Could something else explain it?" Now an AI plays that role. Copy the prompt, fill the brackets, and paste your interpretation **and** the outputs it rests on (tables as text, not your raw data).

```
I am doing exploratory data analysis for a university course. My research question is:
[question]. One row of my data is [unit]; n = [rows].

Here is my interpretation of the relationship between [outcome] and [variable], and the
outputs it is based on:
[your interpretation]
[pasted tables]

Act as a skeptical reviewer. List up to three specific ways this conclusion could be wrong:
confounders, small groups, missing data, measurement problems, or a misleading chart.
For each, say what I could check in the data. Do not rewrite my interpretation.
```

Record the exchange in **H2**. You decide which objections are valid; the AI does not.

---
# Part G · Data story and a verdict on your question (40 min) 🚫 No AI

📑 **Slides:** 07 Wrapping up · "Main chart checklist" · "Caveats worth thinking about" · Data visualization in Python ("combine Seaborn… with Matplotlib for final, detailed refinements")

### G1. Four-sentence summary (80–120 words), as in 08_EDA Task E

1. The dataset: what it contains, sample size, completeness.
2. The most important pattern, with a number.
3. One hypothesis to test next.
4. One key limitation and what it means for interpretation.

Write it as a Python string so the word count is checked for you.

In [ ]:
summary_text = """
(write your four sentences here)
"""
n_words = len(summary_text.split())
print(n_words, "words", "✅" if 80 <= n_words <= 120 else "❌ must be 80-120")

### G2. Hero chart

One chart that carries your main finding. ✔️ **Checklist** (slides: "Main chart checklist"):

- [ ] The **title states the finding** (e.g., "Third-class men had the lowest survival: 14%"), not the variable names.
- [ ] Axes labelled, with units. **Rates (%)**, not raw counts, when groups differ in size.
- [ ] Group sizes (**n**) visible in the chart or caption.
- [ ] Bars sorted or meaningfully ordered. No pie chart with more than 3 slices.
- [ ] **One** highlight colour; everything else muted.
- [ ] Built with Seaborn, refined with Matplotlib.

In [ ]:
# 🎯 Hero chart: build it with Seaborn, refine it with Matplotlib, then save it
fig, ax = plt.subplots(figsize=(8, 4.5))

# sns.___(data=..., x=..., y=..., ax=ax)

ax.set_title("___ <- write the finding, not the variable names ___")
ax.set_xlabel("___")
ax.set_ylabel("___")
plt.tight_layout()
plt.savefig("hero.png", dpi=200, bbox_inches="tight")

### G3. Three Headlines, revisited

Copy the three headlines you wrote in the peer-feedback activity (Part E of that worksheet). What does the data say now?

| # | My headline (copied) | Supported / Contradicted / Cannot tell yet | Evidence (a number, and the Part it comes from) |
| --- | --- | --- | --- |
| 1 | | | |
| 2 | | | |
| 3 | | | |

### G4. Verdict on your research question

Mark one, and explain in 3 to 5 sentences.

- [ ] **Keep as is:** this data can answer it.
- [ ] **Revise:** new version (one sentence, no "and", no "or"): ______
- [ ] **Change data:** this data cannot answer it. What data would?

(write here)

### G5. Next question

One hypothesis to test formally in Topics 3–4 (your limitations are already in sentence 4 of G1):

- H₀: ______
- H₁: ______

---
# Part H · AI log and reflection

Keep the log **as you work**, not at the end. Five to ten rows is typical. If you used no AI, write "No AI used" and complete only H3.

**AI tool(s) and version:** ____________ (e.g., "Claude Sonnet 5.5 via claude.ai", "Gemini in Colab")

### H1. AI log

| # | Part | What I asked (exact prompt, or a link to the shared chat) | What I did with the answer (used / changed / rejected) | How I checked it (ran it, docs page, compared with my output) | What I learned that I could now do without AI |
| --- | --- | --- | --- | --- | --- |
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

### H2. The Skeptic exchange (from Part F)

**Exact prompt:**

(paste here)

**AI response** (unedited; long output can go in an appendix cell at the end):

(paste here)

| AI objection | Valid? (Yes / Partly / No) | My response, or the change I made |
| --- | --- | --- |
| | | |
| | | |

### H3. Reflection (100–150 words) 🚫 No AI

1. **One AI error you caught.** An output that was wrong, outdated, inefficient or superficial. How did you detect it?
2. **Your rule for next time.** One specific personal rule for when you will and will not use AI in data work. ("I will ask for the cause of an error before asking for a fix" is a rule; "I will use AI responsibly" is not.)

(write here)

---
# 📤 Before you submit

- [ ] *Runtime → Restart session and run all* finishes with **no errors**; all outputs visible.
- [ ] Every number in a ✍️ box matches an output above it.
- [ ] Every part has its code, ✍️ box and 🔎 sanity check.
- [ ] The AI log names the tool and version; every AI-assisted part has a row.
- [ ] *File → Download → Download .ipynb*: `EDA_[ProjectShortName]_[LastName].ipynb`, plus `hero.png` (Files panel → right-click → Download).
- [ ] Both files uploaded to Moodle; your *Check-ins* row on the Research Progress Board updated with your G4 verdict and biggest remaining risk.